# Lab C — Roofline & the Interconnect

**Week 2 · GPU Infrastructure for LLMs** · run on the **2× RTX 5090** server (WSL2)

Two ideas from the lecture, made measurable:

1. **The roofline** — every op is either **compute-bound** or **memory-bound**, decided by its *arithmetic intensity* vs the GPU's **ridge point**. You'll measure your 5090's peak compute and peak bandwidth, compute the ridge point, then watch a matrix–vector op sit far below the roof (memory-bound) while a matrix–matrix op hits it (compute-bound).
2. **The interconnect** — your two 5090s have **no NVLink**, only **PCIe 5.0**. You'll measure GPU→GPU bandwidth and see it is ~30–50× slower than on-chip HBM — the reason tensor-parallelism is expensive on this box.

In [ ]:
import torch, time, subprocess, gc
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU{i}: {p.name}  {p.total_memory/1e9:.1f} GB  sm_{p.major}{p.minor}")
def free_gpu(): gc.collect(); torch.cuda.empty_cache()

## 1. Your GPU's two numbers: peak compute & peak bandwidth

- **Peak compute** — how many FP16 multiply-adds per second (measured with a big square matmul).
- **Peak bandwidth** — how fast it moves bytes to/from HBM (measured with a big copy).
- **Ridge point** = peak FLOP/s ÷ peak byte/s, in **FLOP/byte**. Below it → memory-bound; above it → compute-bound.

In [ ]:
def peak_compute_tflops(n=8192, iters=50, warmup=10):
    a = torch.randn(n, n, device="cuda", dtype=torch.float16)
    b = torch.randn(n, n, device="cuda", dtype=torch.float16)
    for _ in range(warmup): c = a @ b
    torch.cuda.synchronize(); t = time.time()
    for _ in range(iters): c = a @ b
    torch.cuda.synchronize()
    dt = (time.time() - t) / iters
    return (2 * n**3) / dt / 1e12    # 2*n^3 FLOPs per matmul

def peak_bandwidth_gbs(nbytes=2_000_000_000, iters=50, warmup=10):
    x = torch.empty(nbytes // 2, dtype=torch.float16, device="cuda")
    for _ in range(warmup): y = x.clone()
    torch.cuda.synchronize(); t = time.time()
    for _ in range(iters): y = x.clone()
    torch.cuda.synchronize()
    dt = (time.time() - t) / iters
    return (2 * nbytes) / dt / 1e9   # read + write

tflops = peak_compute_tflops()
bw_gbs = peak_bandwidth_gbs()
ridge  = (tflops * 1e12) / (bw_gbs * 1e9)
print(f"Peak compute   : {tflops:8.0f} TFLOP/s (FP16)")
print(f"Peak bandwidth : {bw_gbs:8.0f} GB/s")
print(f"Ridge point    : {ridge:8.1f} FLOP/byte   <-- below this = memory-bound, above = compute-bound")

## 2. Arithmetic intensity: matrix–vector vs matrix–matrix

- **Matrix × vector** (like LLM **decode**, batch 1): FLOPs = 2n², bytes read ≈ 2n² → **AI ≈ 1 FLOP/byte** — independent of n, deep in memory-bound territory.
- **Matrix × matrix** (like LLM **prefill**): FLOPs = 2n³, bytes ≈ 2·2n² → **AI ≈ n/2** — climbs with n until it hits the compute roof.

Watch the achieved throughput: matvec will be a tiny fraction of peak; matmul will approach it.

In [ ]:
def achieved_tflops(op, n, iters=50, warmup=10):
    a = torch.randn(n, n, device="cuda", dtype=torch.float16)
    if op == "matvec":
        x = torch.randn(n, 1, device="cuda", dtype=torch.float16); flops = 2 * n * n
        f = lambda: a @ x
    else:
        b = torch.randn(n, n, device="cuda", dtype=torch.float16); flops = 2 * n**3
        f = lambda: a @ b
    for _ in range(warmup): f()
    torch.cuda.synchronize(); t = time.time()
    for _ in range(iters): f()
    torch.cuda.synchronize()
    return flops / ((time.time() - t) / iters) / 1e12

n = 8192
mv = achieved_tflops("matvec", n)
mm = achieved_tflops("matmul", n)
print(f"matrix x vector (AI~1, like decode)   : {mv:8.1f} TFLOP/s  ->  {100*mv/tflops:4.1f}% of peak   [MEMORY-BOUND]")
print(f"matrix x matrix (AI~n/2, like prefill): {mm:8.1f} TFLOP/s  ->  {100*mm/tflops:4.1f}% of peak   [COMPUTE-BOUND]")
free_gpu()

**This is the roofline in one cell:** the matmul uses most of the GPU's compute; the mat-vec wastes almost all of it, bottlenecked on memory. An LLM generating one token at a time is doing mat-vec — that's why decode is slow and batching helps (it turns mat-vec back into mat-mat).

## 3. Prefill vs decode on a real model

Now the same story with an actual LLM: **prefill** (process the whole prompt at once → big matmuls → compute-bound) vs **decode** (one token at a time → mat-vec → memory-bound). Then show that **batching decode** recovers throughput.

In [ ]:
# ===== CONFIG =====
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"   # <-- same model you used in Lab B
# ==================
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map={"": 0})
model.eval()

In [ ]:
prompt = "Explain what a GPU is and why it matters for AI. " * 40      # a long-ish prompt
ids = tok(prompt, return_tensors="pt").to("cuda:0")
n_prompt = ids.input_ids.shape[1]

# --- prefill: one forward pass over the whole prompt ---
torch.cuda.synchronize(); t = time.time()
with torch.no_grad(): _ = model(**ids)
torch.cuda.synchronize(); prefill_s = time.time() - t
print(f"PREFILL : {n_prompt} tokens in {prefill_s*1000:6.1f} ms  ->  {n_prompt/prefill_s:8.0f} tok/s  (compute-bound)")

# --- decode: generate tokens one at a time ---
torch.cuda.synchronize(); t = time.time()
with torch.no_grad():
    out = model.generate(**ids, max_new_tokens=64, do_sample=False)
torch.cuda.synchronize(); decode_s = time.time() - t
n_new = out.shape[1] - n_prompt
print(f"DECODE  : {n_new} tokens in {decode_s*1000:6.1f} ms  ->  {n_new/decode_s:8.1f} tok/s  (memory-bound)")
print(f"\nDecode is ~{(n_prompt/prefill_s)/(n_new/decode_s):.0f}x slower per token than prefill — same weights, different arithmetic intensity.")

In [ ]:
# --- batching decode: amortize the weight reads across many sequences ---
print(f"{'batch':>6} | {'total tok/s':>12}")
for B in [1, 4, 16]:
    batch = tok([prompt] * B, return_tensors="pt", padding=True).to("cuda:0")
    torch.cuda.synchronize(); t = time.time()
    with torch.no_grad():
        out = model.generate(**batch, max_new_tokens=32, do_sample=False)
    torch.cuda.synchronize(); dt = time.time() - t
    new = (out.shape[1] - batch.input_ids.shape[1]) * B
    print(f"{B:>6} | {new/dt:>12.0f}")
    free_gpu()
print("\nHigher batch -> more tokens/sec from the SAME weight reads. This is why servers batch requests.")

## 4. The interconnect: PCIe vs HBM (the no-NVLink reality)

First confirm the topology (expect `PHB`/`NODE`/`SYS`, **not** `NV#`), then measure how fast the two cards can actually exchange data.

In [ ]:
print(subprocess.run(["nvidia-smi", "topo", "-m"], capture_output=True, text=True).stdout)

In [ ]:
if torch.cuda.device_count() < 2:
    print("Only one GPU visible — skip the P2P test (need both 5090s).")
else:
    print("Can GPU0 access GPU1 directly (P2P)? ->", torch.cuda.can_device_access_peer(0, 1))
    nbytes = 2_000_000_000
    x = torch.empty(nbytes // 2, dtype=torch.float16, device="cuda:0")
    y = torch.empty(nbytes // 2, dtype=torch.float16, device="cuda:1")
    for _ in range(5): y.copy_(x)                    # warmup
    torch.cuda.synchronize(); t = time.time()
    for _ in range(20): y.copy_(x)
    torch.cuda.synchronize(); dt = (time.time() - t) / 20
    p2p_gbs = nbytes / dt / 1e9
    print(f"\nGPU0 -> GPU1 over PCIe : {p2p_gbs:7.0f} GB/s")
    print(f"On-chip HBM bandwidth  : {bw_gbs:7.0f} GB/s  (from Section 1)")
    print(f"HBM is ~{bw_gbs/p2p_gbs:.0f}x faster than the link between the two cards.")
    del x, y; free_gpu()

## 5. Implication: why not tensor-parallel on this box?

Tensor parallelism splits every layer across both GPUs and must **all-reduce activations every layer** over that slow PCIe link. Let's estimate the tax.

In [ ]:
# rough cost of moving one layer's activations across the link during tensor-parallel
try:
    link_gbs = p2p_gbs
except NameError:
    link_gbs = 55.0   # fallback: typical PCIe 5.0 x16, GB/s
hidden, seq, batch, bytes_ = 4096, 2048, 1, 2
act_bytes = hidden * seq * batch * bytes_
per_layer_ms = (act_bytes / (link_gbs * 1e9)) * 1e3
print(f"~{act_bytes/1e6:.1f} MB of activations per layer, per all-reduce")
print(f"~{per_layer_ms:.2f} ms just in communication per layer over the link")
print(f"For a 32-layer model that's ~{per_layer_ms*32:.1f} ms/token of PURE communication — often more than the compute.")
print("\n=> On a no-NVLink box: prefer DATA parallelism (a full model per card) or PIPELINE parallelism.")
print("   Save tensor parallelism for NVLink/NVSwitch machines.")

## Reflection (write your answers)

1. What is your 5090's measured **ridge point** (FLOP/byte)? Higher or lower than the ~500 from the lecture example?
2. What % of peak did **mat-vec** vs **mat-mul** reach? Which is decode, which is prefill?
3. How many **× faster** was decode when you went from batch 1 → 16? Why isn't it 16×?
4. What GPU0→GPU1 bandwidth did you measure, and how many times slower is it than HBM? Does `topo -m` confirm there's no NVLink?

### Cleanup

In [ ]:
for name in ["model", "out", "ids", "batch"]:
    if name in dir():
        try: exec(f"del {name}")
        except Exception: pass
free_gpu()
print("done")